<a href="https://colab.research.google.com/github/MathewBiddle/map-of-activities/blob/main/MBON_harvest_registration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install selenium beautifulsoup4 requests

In [2]:
## For google spreadsheet reading you need to authenticate w/ google

import pandas as pd

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [3]:
def get_sheet_data(url: str, sheet: str) -> pd.DataFrame:
  '''
  Gets data from specific worksheet in Google Spreadsheet and
  returns as a DataFrame.

  parameters:
  sheet (str): name of worksheet, eg 'Form Responses 1'

  returns:
  DataFrame: worksheet data as a DataFrame

  '''

  worksheet = gc.open_by_url(url)
  room_n = worksheet.worksheet(sheet)

  df = pd.DataFrame(room_n.get_all_records())

  return df

In [4]:
url = 'https://docs.google.com/spreadsheets/d/1jBS8ASS27yV8APZ8Fh-tgX6dHdopwianrUZv0kbKcxw/edit?gid=1698140136#gid=1698140136'
sheet = 'Form Responses 1'

df = get_sheet_data(url, sheet)

df = df.replace('(?i)yes$',True, regex=True).replace('(?i)no$',False,regex=True)

df.sample(4)

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,Upload the dataset.,Are there scripts or code used to process the data?,"If yes to above, and they are publicly available, please include appropriate link(s) here.","If yes to above, and the code is available, please include appropriate link(s) here.",Would you like this dataset visualized in the MBON Data Portal (https://mbon.ioos.us/)?,Email Address,"If the dataset is already visualized in the MBON data portal, please include the link(s) to the data layer(s) here.",,What is the expected timeline for this dataset?,additonal comments
27,10/19/2021 14:22,Santa Barbara Channel Coastal and Island fish ...,citizen scientist monitoring data for rocky re...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,,,,,
9,9/10/2024 15:14:29,FKNMS Cruises CTD data,"Bathymetry, nutrients, and more collected by t...",,Ian Smith,Ian.Smith@noaa.gov,Ian Smith,Ian.Smith@noaa.gov,SE US,GCOOS,...,,True,https://github.com/USF-IMARS/seus-mbon-cruise-...,,True,murray.tylar@gmail.com,,,,
35,10/19/2021 14:22,Abundance and species composition of benthic h...,counts of heterobranch molluscs (sea slugs and...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,,,,,,
54,8/5/2021 16:00:15,Benthic Epifauna Biomass and Abundance Data in...,This dataset contains benthic epifauna biomass...,,False,adrienne@axiomdatascience.com,Katrin Iken,kbiken@alaska.edu,Arctic,AOOS,...,,True,,,True,,https://mbon.ioos.us/#metadata/9649bae5-3020-4...,,,


## Function to extract json-ld from webpage

In [5]:
from selenium import webdriver
from bs4 import BeautifulSoup
import json
import requests

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')  # Run Chrome in headless mode
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

def get_ld_json(url: str) -> dict:
    parser = "lxml"
    req = requests.get(url)
    soup = BeautifulSoup(req.text, parser)
    return json.loads("".join(soup.find("script", {"type":"application/ld+json"}).contents))

## Check through datasets that have a link to a repository and extract json-ld

In [6]:
df['spatialCoverage'] = pd.Series()

for index, row in df.loc[df['If yes to above, please include appropriate link(s) here.']!='',['If yes to above, please include appropriate link(s) here.','Dataset title']].iterrows():

  url = row['If yes to above, please include appropriate link(s) here.']
  title = row['Dataset title']


  if url.startswith('10.154'):
    url = f'https://dx.doi.org/{url}'
    #print(f'{url}\n')

  elif '\ndata can also be accessed by using the repository\'s API.' in url:
    url = url.replace('data can also be accessed by using the repository\'s API.','')
    #print(f'{url}\n')

  elif url.endswith('.pdf'):
    continue

  elif 'usf.box.com' in url:
    continue

  elif 'neracoos.org/erddap' in url:
    # NERACOOS is running an old ERDDAP which represents
    # schema.org bounding box incorrectly. See
    # https://erddap.github.io/changes#version-200
    continue

  try:
    spatial = get_ld_json(url)['spatialCoverage']['geo']
    print(f'{url} has spatial {spatial}')
    df.loc[df['Dataset title'] == title, 'spatialCoverage'] = [spatial]

  except:

    try:

      browser = webdriver.Chrome(options=chrome_options)
      browser.get(url)
      html_source = browser.page_source
      soup = BeautifulSoup(html_source, 'lxml')
      spatial = json.loads("".join(soup.find("script", {"type":"application/ld+json"}).contents))
      spatial = spatial['spatialCoverage']['geo']
      df.loc[df['Dataset title'] == title, 'spatialCoverage'] = [spatial]

    except:
      print(f'{url} cant find json-ld')

https://dx.doi.org/10.15468/bfd6ci cant find json-ld
https://dx.doi.org/10.15468/h585qq has spatial {'@type': 'GeoShape', 'box': '24.647 26.785 -82.705 -79.277'}
https://grunt.sefsc.noaa.gov/rvc_analysis20/samples/index cant find json-ld
https://ecotaxa.obs-vlfr.fr/prj/9989 cant find json-ld
https://data.piscoweb.org/metacatui/view/doi%3A10.6085%2FAA%2Fmarine_cbs.5.6 cant find json-ld
https://pubmed.ncbi.nlm.nih.gov/29937700/ cant find json-ld
https://dx.doi.org/10.15468/buqg4u  has spatial {'@type': 'GeoShape', 'box': '24.476 25.006 -81.715 -80.379'}
https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=5 has spatial [{'@type': 'GeoShape', 'box': '32.8 -120.6344833 34.87315 -118.4'}]
https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=3 has spatial [{'@type': 'GeoShape', 'box': '32.8 -120.65022 34.87315 -118.4'}]
https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=7 has spatial [{'@type': 'GeoShape', 'box': '32.8 -120.65022 34.87315 -118.4

## Extract coordinate information

Add coordinate information from schema.org back into the DataFrame.

In [7]:
df = pd.concat([df, pd.json_normalize(df['spatialCoverage'])], axis=1)

df.loc[~df['spatialCoverage'].isna()]

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,"If the dataset is already visualized in the MBON data portal, please include the link(s) to the data layer(s) here.",,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,polygon
2,9/10/2024 15:27:50,South Florida Fisheries Habitat Assessment (FW...,taxa occurrence. Six seagrass species; ~36k re...,,Luke McEachron,Lucas.McEachron@myfwc.com,Luke McEachron,Lucas.McEachron@myfwc.com,SE US,,...,,,,,"{'@type': 'GeoShape', 'box': '24.647 26.785 -8...",GeoShape,24.647 26.785 -82.705 -79.277,NaN,NaN,NaN
21,6/20/2023 13:47:54,Time series of zooplankton abundance in South ...,Sampling are carried out bi-monthly on the R/V...,,"Digna Rueda-Roa (College of Marine Science, Un...",druedaro@usf.edu,Tylar Murray / Sebastian DiGeronimo (College ...,tylarmurray@usf.edu / sebastian15@usf.edu,SE US,GCOOS,...,,,"Please, put this in hold. \nWe uploaded a part...",,"{'@type': 'GeoShape', 'box': '24.476 25.006 -8...",GeoShape,24.476 25.006 -81.715 -80.379,NaN,NaN,NaN
23,10/19/2021 14:22,Santa Barbara Channel Marine BON: Nearshore ke...,This dataset contains counts of epibenthic alg...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.634483...",GeoShape,32.8 -120.6344833 34.87315 -118.4,NaN,NaN,NaN
24,10/19/2021 14:22,Southern California Bight Marine BON: Integrat...,This dataset contains cover of kelp forest ses...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,NaN
25,10/19/2021 14:22,Southern California Bight Marine BON: cummulat...,Dataset contains all species from datasets of ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,NaN
26,10/19/2021 14:22,SBC LTER: Time series of quarterly NetCDF file...,This data is a time series of canopy area of g...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '27.01 -124.77 48...",GeoShape,27.01 -124.77 48.40 -114.04,NaN,NaN,NaN
27,10/19/2021 14:22,Santa Barbara Channel Coastal and Island fish ...,citizen scientist monitoring data for rocky re...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '34.4030 -120.066...",GeoShape,34.4030 -120.0669 34.4611 -119.8660,NaN,NaN,NaN
29,10/19/2021 14:22,Santa Barbara Channel fish surveys at deep ree...,fish surveys from deep natural reefs in the no...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '33.91 -119.52 34...",GeoShape,33.91 -119.52 34.01 -119.45,NaN,NaN,NaN
31,10/19/2021 14:22,NWFSC fish and invertebrate diversity derived ...,This dataset presents the community structure ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,,"{'@type': 'GeoShape', 'box': '31.9 -121.15 34....",GeoShape,31.9 -121.15 34.5 -117.27,NaN,NaN,NaN
32,10/19/2021 14:22,Plumes and Blooms: Curated oceanographic and p...,"The Plumes and Bloom

## Box

In [8]:
import geopandas as gpd

test = df.loc[~df['box'].isna()]

# read from 'box'
def box_to_wkt(box_str):
       """Converts a box string to WKT format."""
       try:
           # Assuming box_str is in the format 'north, west, south, east'
           south, west, north, east = map(float, box_str.split(' '))
           # Create WKT polygon string
           wkt_polygon = f'POLYGON(({west} {north}, {east} {north}, {east} {south}, {west} {south}, {west} {north}))'
           return wkt_polygon
       except (ValueError, AttributeError):
           # Handle cases where box_str is not in the expected format or is None
           return None

test['wkt'] = test['box'].apply(box_to_wkt)

#test['geometry'] = gpd.GeoSeries.from_wkt(test['wkt'], crs='EPSG:4326')

#test

df.loc[~df['box'].isna(),'wkt'] = test['wkt']

df.loc[~df['box'].isna()]
# read 'box' into geometry somehow
#gpd.GeoSeries.from_wkt(df['box'])

<ipython-input-8-65551a2ccba0>:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['wkt'] = test['box'].apply(box_to_wkt)


,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,polygon,wkt
2,9/10/2024 15:27:50,South Florida Fisheries Habitat Assessment (FW...,taxa occurrence. Six seagrass species; ~36k re...,,Luke McEachron,Lucas.McEachron@myfwc.com,Luke McEachron,Lucas.McEachron@myfwc.com,SE US,,...,,,,"{'@type': 'GeoShape', 'box': '24.647 26.785 -8...",GeoShape,24.647 26.785 -82.705 -79.277,NaN,NaN,NaN,"POLYGON((26.785 -82.705, -79.277 -82.705, -79...."
21,6/20/2023 13:47:54,Time series of zooplankton abundance in South ...,Sampling are carried out bi-monthly on the R/V...,,"Digna Rueda-Roa (College of Marine Science, Un...",druedaro@usf.edu,Tylar Murray / Sebastian DiGeronimo (College ...,tylarmurray@usf.edu / sebastian15@usf.edu,SE US,GCOOS,...,,"Please, put this in hold. \nWe uploaded a part...",,"{'@type': 'GeoShape', 'box': '24.476 25.006 -8...",GeoShape,24.476 25.006 -81.715 -80.379,NaN,NaN,NaN,"POLYGON((25.006 -81.715, -80.379 -81.715, -80...."
23,10/19/2021 14:22,Santa Barbara Channel Marine BON: Nearshore ke...,This dataset contains counts of epibenthic alg...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.634483...",GeoShape,32.8 -120.6344833 34.87315 -118.4,NaN,NaN,NaN,"POLYGON((-120.6344833 34.87315, -118.4 34.8731..."
24,10/19/2021 14:22,Southern California Bight Marine BON: Integrat...,This dataset contains cover of kelp forest ses...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,..."
25,10/19/2021 14:22,Southern California Bight Marine BON: cummulat...,Dataset contains all species from datasets of ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,..."
26,10/19/2021 14:22,SBC LTER: Time series of quarterly NetCDF file...,This data is a time series of canopy area of g...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '27.01 -124.77 48...",GeoShape,27.01 -124.77 48.40 -114.04,NaN,NaN,NaN,"POLYGON((-124.77 48.4, -114.04 48.4, -114.04 2..."
27,10/19/2021 14:22,Santa Barbara Channel Coastal and Island fish ...,citizen scientist monitoring data for rocky re...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '34.4030 -120.066...",GeoShape,34.4030 -120.0669 34.4611 -119.8660,NaN,NaN,NaN,"POLYGON((-120.0669 34.4611, -119.866 34.4611, ..."
29,10/19/2021 14:22,Santa Barbara Channel fish surveys at deep ree...,fish surveys from deep natural reefs in the no...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,,"{'@type': 'GeoShape', 'box': '33.91 -119.52 34...",GeoShape,33.91 -119.52 34.01 -119.45,NaN,NaN,NaN,"POLYGON((-119.52 34.01, -119.45 34.01, -119.45..."
31,10/19/2021 14:22,NWFSC fish and invertebrate diversity derived ...,This dataset presents the community structure ...,,"Information Manager, Southern Calif

In [9]:
df.loc[~df['box'].isna(),'box']

,box
2,24.647 26.785 -82.705 -79.277
21,24.476 25.006 -81.715 -80.379
23,32.8 -120.6344833 34.87315 -118.4
24,32.8 -120.65022 34.87315 -118.4
25,32.8 -120.65022 34.87315 -118.4
26,27.01 -124.77 48.40 -114.04
27,34.4030 -120.0669 34.4611 -119.8660
29,33.91 -119.52 34.01 -119.45
31,31.9 -121.15 34.5 -117.27
32,33.3 -79.21 33.38 -79.17


## Polygon

In [10]:
temp=df.loc[~df['polygon'].isna()]

temp['wkt'] = 'POLYGON ((' + temp['polygon'].astype(str) + '))'

df.loc[~df['polygon'].isna(),'wkt'] = temp['wkt']

# gs = gpd.GeoSeries.from_wkt(temp['wkt'])

# gdf2 = gpd.GeoDataFrame(
#     temp,
#     geometry=gs,
#     crs='EPSG:4326'
#     )

# gdf.loc[~gdf['polygon'].isna(),'geometry'] = gdf2['geometry']


df.loc[~df['polygon'].isna()]
#gdf.merge(gdf2, how='inner', on='Dataset title')

<ipython-input-10-01eb75825acc>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp['wkt'] = 'POLYGON ((' + temp['polygon'].astype(str) + '))'


,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,polygon,wkt
51,8/9/2021 16:46:29,Time series of zooplankton abundance of the So...,Sampling is carried out bi-monthly on the R/V ...,,Enrique Montes (College of Marine Science of t...,emontesh@usf.edu,Tylar Murray (College of Marine Science of the...,tylarmurray@usf.edu,SE US,GCOOS,...,,,5/3/2022: put this on hold because all we can ...,"{'@type': 'GeoShape', 'polygon': '-81.717 24.4...",GeoShape,NaN,NaN,NaN,"-81.717 24.478,-81.717 25.35167,-80.38 25.3516...","POLYGON ((-81.717 24.478,-81.717 25.35167,-80...."


## GeoCoordinates

In [11]:
import geopandas as gpd
temp = df.loc[df['@type']=='GeoCoordinates', ['latitude','longitude']]


gdf = gpd.GeoDataFrame(
    temp, geometry=gpd.points_from_xy(
        temp['longitude'],
        temp['latitude'])
    )

df.loc[df['@type']=='GeoCoordinates','geometry'] = gdf['geometry']

df.loc[df['@type']=='GeoCoordinates']

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,polygon,wkt,geometry
39,10/19/2021 14:22,Data to support manuscript: A Comparison of Tw...,fish dataset was collected at platform Harmony...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoCoordinates', 'latitude': '34.37...",GeoCoordinates,NaN,34.37,-120.16,NaN,NaN,POINT (-120.16 34.37)
41,10/19/2021 14:22,Santa Barbara Channel Marine BON: Gray Whales ...,dataset documents the passage of gray whales (...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoCoordinates', 'latitude': '34.40...",GeoCoordinates,NaN,34.40766,-119.877969,NaN,NaN,POINT (-119.87797 34.40766)


In [12]:
# wkt to geometry

temp = df.loc[~df['wkt'].isna()]

gs = gpd.GeoSeries.from_wkt(temp['wkt'])

gdf2 = gpd.GeoDataFrame(
    temp,
    geometry=gs,
    crs='EPSG:4326'
    )

df.loc[~df['wkt'].isna(),'geometry'] = gdf2['geometry']

df.loc[~df['wkt'].isna()]

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,polygon,wkt,geometry
2,9/10/2024 15:27:50,South Florida Fisheries Habitat Assessment (FW...,taxa occurrence. Six seagrass species; ~36k re...,,Luke McEachron,Lucas.McEachron@myfwc.com,Luke McEachron,Lucas.McEachron@myfwc.com,SE US,,...,,,"{'@type': 'GeoShape', 'box': '24.647 26.785 -8...",GeoShape,24.647 26.785 -82.705 -79.277,NaN,NaN,NaN,"POLYGON((26.785 -82.705, -79.277 -82.705, -79....","POLYGON ((26.785 -82.705, -79.277 -82.705, -79..."
21,6/20/2023 13:47:54,Time series of zooplankton abundance in South ...,Sampling are carried out bi-monthly on the R/V...,,"Digna Rueda-Roa (College of Marine Science, Un...",druedaro@usf.edu,Tylar Murray / Sebastian DiGeronimo (College ...,tylarmurray@usf.edu / sebastian15@usf.edu,SE US,GCOOS,...,"Please, put this in hold. \nWe uploaded a part...",,"{'@type': 'GeoShape', 'box': '24.476 25.006 -8...",GeoShape,24.476 25.006 -81.715 -80.379,NaN,NaN,NaN,"POLYGON((25.006 -81.715, -80.379 -81.715, -80....","POLYGON ((25.006 -81.715, -80.379 -81.715, -80..."
23,10/19/2021 14:22,Santa Barbara Channel Marine BON: Nearshore ke...,This dataset contains counts of epibenthic alg...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoShape', 'box': '32.8 -120.634483...",GeoShape,32.8 -120.6344833 34.87315 -118.4,NaN,NaN,NaN,"POLYGON((-120.6344833 34.87315, -118.4 34.8731...","POLYGON ((-120.63448 34.87315, -118.4 34.87315..."
24,10/19/2021 14:22,Southern California Bight Marine BON: Integrat...,This dataset contains cover of kelp forest ses...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,...","POLYGON ((-120.65022 34.87315, -118.4 34.87315..."
25,10/19/2021 14:22,Southern California Bight Marine BON: cummulat...,Dataset contains all species from datasets of ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,...","POLYGON ((-120.65022 34.87315, -118.4 34.87315..."
26,10/19/2021 14:22,SBC LTER: Time series of quarterly NetCDF file...,This data is a time series of canopy area of g...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoShape', 'box': '27.01 -124.77 48...",GeoShape,27.01 -124.77 48.40 -114.04,NaN,NaN,NaN,"POLYGON((-124.77 48.4, -114.04 48.4, -114.04 2...","POLYGON ((-124.77 48.4, -114.04 48.4, -114.04 ..."
27,10/19/2021 14:22,Santa Barbara Channel Coastal and Island fish ...,citizen scientist monitoring data for rocky re...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoShape', 'box': '34.4030 -120.066...",GeoShape,34.4030 -120.0669 34.4611 -119.8660,NaN,NaN,NaN,"POLYGON((-120.0669 34.4611, -119.866 34.4611, ...","POLYGON ((-120.0669 34.4611, -119.866 34.4611,..."
29,10/19/2021 14:22,Santa Barbara Channel fish surveys at deep ree...,fish surveys from deep natural reefs in the no...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbo

In [13]:
df.loc[~df['@type'].isna()]

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,polygon,wkt,geometry
2,9/10/2024 15:27:50,South Florida Fisheries Habitat Assessment (FW...,taxa occurrence. Six seagrass species; ~36k re...,,Luke McEachron,Lucas.McEachron@myfwc.com,Luke McEachron,Lucas.McEachron@myfwc.com,SE US,,...,,,"{'@type': 'GeoShape', 'box': '24.647 26.785 -8...",GeoShape,24.647 26.785 -82.705 -79.277,NaN,NaN,NaN,"POLYGON((26.785 -82.705, -79.277 -82.705, -79....","POLYGON ((26.785 -82.705, -79.277 -82.705, -79..."
21,6/20/2023 13:47:54,Time series of zooplankton abundance in South ...,Sampling are carried out bi-monthly on the R/V...,,"Digna Rueda-Roa (College of Marine Science, Un...",druedaro@usf.edu,Tylar Murray / Sebastian DiGeronimo (College ...,tylarmurray@usf.edu / sebastian15@usf.edu,SE US,GCOOS,...,"Please, put this in hold. \nWe uploaded a part...",,"{'@type': 'GeoShape', 'box': '24.476 25.006 -8...",GeoShape,24.476 25.006 -81.715 -80.379,NaN,NaN,NaN,"POLYGON((25.006 -81.715, -80.379 -81.715, -80....","POLYGON ((25.006 -81.715, -80.379 -81.715, -80..."
23,10/19/2021 14:22,Santa Barbara Channel Marine BON: Nearshore ke...,This dataset contains counts of epibenthic alg...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoShape', 'box': '32.8 -120.634483...",GeoShape,32.8 -120.6344833 34.87315 -118.4,NaN,NaN,NaN,"POLYGON((-120.6344833 34.87315, -118.4 34.8731...","POLYGON ((-120.63448 34.87315, -118.4 34.87315..."
24,10/19/2021 14:22,Southern California Bight Marine BON: Integrat...,This dataset contains cover of kelp forest ses...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,...","POLYGON ((-120.65022 34.87315, -118.4 34.87315..."
25,10/19/2021 14:22,Southern California Bight Marine BON: cummulat...,Dataset contains all species from datasets of ...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoShape', 'box': '32.8 -120.65022 ...",GeoShape,32.8 -120.65022 34.87315 -118.4,NaN,NaN,NaN,"POLYGON((-120.65022 34.87315, -118.4 34.87315,...","POLYGON ((-120.65022 34.87315, -118.4 34.87315..."
26,10/19/2021 14:22,SBC LTER: Time series of quarterly NetCDF file...,This data is a time series of canopy area of g...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoShape', 'box': '27.01 -124.77 48...",GeoShape,27.01 -124.77 48.40 -114.04,NaN,NaN,NaN,"POLYGON((-124.77 48.4, -114.04 48.4, -114.04 2...","POLYGON ((-124.77 48.4, -114.04 48.4, -114.04 ..."
27,10/19/2021 14:22,Santa Barbara Channel Coastal and Island fish ...,citizen scientist monitoring data for rocky re...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbon@msi.ucsb.edu,Southern California Bight,SCCOOS,...,,,"{'@type': 'GeoShape', 'box': '34.4030 -120.066...",GeoShape,34.4030 -120.0669 34.4611 -119.8660,NaN,NaN,NaN,"POLYGON((-120.0669 34.4611, -119.866 34.4611, ...","POLYGON ((-120.0669 34.4611, -119.866 34.4611,..."
29,10/19/2021 14:22,Santa Barbara Channel fish surveys at deep ree...,fish surveys from deep natural reefs in the no...,,"Information Manager, Southern California Bight...",sbcbon@msi.ucsb.edu,Li Kui,sbcbo

In [14]:
!pip install folium matplotlib mapclassify

In [25]:
gdf = gpd.GeoDataFrame(df, geometry=df['geometry'], crs = 'epsg:4326')

gdf

,Timestamp,Dataset title,Dataset summary,Crossfunded? (Yes/No),Who is the data management POC for the dataset?,Data management POC email:,Who is the technical POC for the dataset?,Technical POC email:,Which MBON project is this dataset associated with?,"If you have worked with a Regional Association, please indicate which one(s).",...,What is the expected timeline for this dataset?,additonal comments,spatialCoverage,@type,box,latitude,longitude,polygon,wkt,geometry
0,9/10/2024 15:30:54,Marine Invertebrate Voucher Specimens (FWC-Col...,taxa occurrence. 1950-present,,Luke McEachron,Lucas.McEachron@myfwc.com,Luke McEachron,Lucas.McEachron@myfwc.com,SE US,,...,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,9/10/2024 15:29:13,SEMAP-South Atlantic trawl surveys,1983-Present. conversion to DarwinCore in prog...,,"Jennifer Dorton, Kyle Wilcox","jdorton@secoora.org, Kyle@axiomdatascience.com","Jennifer Dorton, Kyle Wilcox","jdorton@secoora.org, Kyle@axiomdatascience.com",SE US,,...,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,9/10/2024 15:27:50,South Florida Fisheries Habitat Assessment (FW...,taxa occurrence. Six seagrass species; ~36k re...,,Luke McEachron,Lucas.McEachron@myfwc.com,Luke McEachron,Lucas.McEachron@myfwc.com,SE US,,...,,,"{'@type': 'GeoShape', 'box': '24.647 26.785 -8...",GeoShape,24.647 26.785 -82.705 -79.277,NaN,NaN,NaN,"POLYGON((26.785 -82.705, -79.277 -82.705, -79....","POLYGON ((26.785 -82.705, -79.277 -82.705, -79..."
3,9/10/2024 15:25:10,FKNMS WS Pigment Phytoplankton,generation of taxa output in progress.,,Sebastian,sebastian15@usf.edu,Sebastian,sebastian15@usf.edu,SE US,,...,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,9/10/2024 15:23:24,Walton Smith Primary Productivity,On Ian Smith's desktop,,Ian Smith,Ian.Smith@noaa.gov,Ian Smith,Ian.Smith@noaa.gov,SE US,,...,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,,"Marine Bird Sighting Data, Arctic Marine Biodi...",,,,,Adrienne Canino @ Axiom,,Arctic,AOOS,...,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
60,,Vessel line-transect surveys of Arctic cetacea...,,,,,Adrienne Canino @ Axiom,,Arctic,AOOS,...,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
61,,Vessel line-transect surveys of Arctic pinnipe...,,,,,Adrienne Canino @ Axiom,,Arctic,AOOS,...,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
62,,Vessel line-transect surveys of Arctic marine ...,,,,,Adrienne Canino @ Axiom,,Arctic,AOOS,...,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [26]:
gdf.columns.tolist()

['Timestamp',
 'Dataset title',
 'Dataset summary',
 'Crossfunded? (Yes/No)',
 'Who is the data management POC for the dataset? ',
 'Data management POC email:',
 'Who is the technical POC for the dataset? ',
 'Technical POC email:',
 'Which MBON project is this dataset associated with?',
 'If you have worked with a Regional Association, please indicate which one(s).',
 'Are there any deadlines associated with this dataset?',
 'If one exists, enter the DOI for the dataset.',
 'If one exists, enter a citation for the dataset.',
 'Are the data accessible via the web?',
 'If yes to above, please include appropriate link(s) here.',
 'Has the dataset been loaded into ERDDAP?',
 'If yes to above, please include appropriate ERDDAP link(s) here.',
 'Has the dataset been translated into DarwinCore?',
 'Has the dataset been submitted to OBIS?',
 'If yes to above, please include appropriate OBIS link(s) here.',
 'Has the dataset been archived at NCEI?',
 'If yes to above, please include appropria

In [27]:
gdf[(gdf['Dataset title']!= 'South Florida Fisheries Habitat Assessment (FWC-Seagrass)') & (gdf['Dataset title']!= 'Time series of zooplankton abundance in South Florida from 2015 onward (MBON program)')].explore(tooltip=['Dataset title','spatialCoverage'], popup=['Dataset title','spatialCoverage','wkt','If yes to above, please include appropriate link(s) here.'])

In [18]:
gdf.loc[gdf['Dataset title'] == 'South Florida Fisheries Habitat Assessment (FWC-Seagrass)',['box','wkt','If yes to above, please include appropriate link(s) here.']]

,box,wkt,"If yes to above, please include appropriate link(s) here."
2,24.647 26.785 -82.705 -79.277,"POLYGON((26.785 -82.705, -79.277 -82.705, -79....",10.15468/h585qq


In [19]:
gdf.loc[gdf['Dataset title'] == 'Time series of zooplankton abundance in South Florida from 2015 onward (MBON program)', ['box','wkt','If yes to above, please include appropriate link(s) here.']]

,box,wkt,"If yes to above, please include appropriate link(s) here."
21,24.476 25.006 -81.715 -80.379,"POLYGON((25.006 -81.715, -80.379 -81.715, -80....",10.15468/buqg4u


In [20]:
gdf.loc[gdf['Dataset title'] == 'SBC LTER: Time series of quarterly NetCDF files of kelp biomass in the canopy from Landsat 5, 7 and 8, since 1984 (ongoing)', ['box','wkt','If yes to above, please include appropriate link(s) here.']]

,box,wkt,"If yes to above, please include appropriate link(s) here."
26,27.01 -124.77 48.40 -114.04,"POLYGON((-124.77 48.4, -114.04 48.4, -114.04 2...",https://portal.edirepository.org/nis/mapbrowse...


In [21]:
df['If yes to above, please include appropriate link(s) here.'].unique()

array(['10.15468/bfd6ci', '', '10.15468/h585qq',
       'https://safmc.net/wp-content/uploads/2022/05/SG_A1a_EwE-Report_Dec2021.pdf',
       'https://usf.box.com/s/dvoi1ve0jn3apbdlad114uhn0pvmjool',
       'https://grunt.sefsc.noaa.gov/rvc_analysis20/samples/index',
       'https://ecotaxa.obs-vlfr.fr/prj/9989',
       'https://data.piscoweb.org/metacatui/view/doi%3A10.6085%2FAA%2Fmarine_cbs.5.6',
       'https://pubmed.ncbi.nlm.nih.gov/29937700/', '10.15468/buqg4u ',
       'http://www.neracoos.org/erddap/info/WBTS_CFIN_2004_2017/index.html',
       'https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=5',
       'https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=3',
       'https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=7',
       'https://portal.edirepository.org/nis/mapbrowse?scope=knb-lter-sbc&identifier=74',
       'https://portal.edirepository.org/nis/mapbrowse?scope=edi&identifier=141',
       'https://portal.edirepository.o

In [22]:
# script =<script src="https://code.jquery.com/jquery-3.6.0.slim.min.js" integrity="sha256-u7e5khyithlIdTpu22PHhENmPcRdFiHRjhAuHcs05RI=" crossorigin="anonymous"></script>
#         <script type="text/javascript" src="https://cdn.datatables.net/1.11.5/js/jquery.dataTables.min.js"></script>
#         <script>
#         $(document).ready( function () {
#             $('#table').DataTable();
#         } );
#         let table = new DataTable('#table');
#         </script>